# GPT Transformer (next token prediction)

In [6]:
import torch
import torch.nn as nn
from typing import Tuple
from torch.nn import functional as F
from dataclasses import dataclass


@dataclass
class GPTConfig:
    batch_size: int = 16
    context_len: int = 256
    n_embd: int = 384
    n_head: int = 6
    n_layer: int = 6
    vocab_size: int = 50257  # GPT-2 default
    device: str = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.n_head = config.n_head
        self.n_embd = config.n_embd

        # Combined Q, K, V projection
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=False)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd)

    def forward(self, x, external_mask=None):
        batch_size, seq_len, n_embd = x.size()

        # Project and split into q, k, v
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)

        # Reshape for multi-head: (batch_size, n_head, seq_len, head_size)
        head_size = n_embd // self.n_head
        q = q.view(batch_size, seq_len, self.n_head, head_size).transpose(1, 2)
        k = k.view(batch_size, seq_len, self.n_head, head_size).transpose(1, 2)
        v = v.view(batch_size, seq_len, self.n_head, head_size).transpose(1, 2)

        # --- THE MASKING LOGIC ---
        if external_mask is not None:
            # DYNAMIC MASK: Used for padding variable-length sequences
            # PyTorch's SDPA will combine this mask with causal logic if needed
            y = F.scaled_dot_product_attention(
                q,
                k,
                v,
                attn_mask=external_mask,
                is_causal=True,  # Still causal, but respects the padding
            )
        else:
            # STATIC MASK: Blazing fast, used for packed/fixed sequences
            y = F.scaled_dot_product_attention(
                q,
                k,
                v,
                attn_mask=None,
                is_causal=True,
            )

        y = y.transpose(1, 2).contiguous().view(batch_size, seq_len, n_embd)
        return self.c_proj(y)

In [ ]:
class ExplicitAttention(nn.Module):
    """FOR LEARNING PURPOSES

    :param nn: _description_
    """

    def __init__(self, config: GPTConfig):
        super().__init__()
        self.n_head = config.n_head
        self.n_embd = config.n_embd

        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=False)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd)

        # We still need the static causal mask buffer for the "don't look at future" rule
        self.register_buffer(
            "causal_mask",
            torch.tril(torch.ones(config.context_len, config.context_len)).view(
                1,
                1,
                config.context_len,
                config.context_len,
            ),
        )

    def forward(self, x, external_mask=None):
        batch_size, seq_len, n_embd = x.size()
        head_size = n_embd // self.n_head

        # 1. Project to Q, K, V
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)

        # 2. Reshape for Multi-Head: (batch_size, n_head, seq_len, head_size)
        q = q.view(batch_size, seq_len, self.n_head, head_size).transpose(1, 2)
        k = k.view(batch_size, seq_len, self.n_head, head_size).transpose(1, 2)
        v = v.view(batch_size, seq_len, self.n_head, head_size).transpose(1, 2)

        # 3. Calculate raw attention scores (scaled dot-product)
        # (batch, n_head, seq_len, head_size) @ (batch, n_head, head_size, seq_len) -> (batch, n_head, seq_len, seq_len)
        att = (q @ k.transpose(-2, -1)) * (1.0 / (head_size**0.5))

        # 4. APPLY CAUSAL MASK (Static)
        # We slice the buffer to the current sequence length (T x T)
        att = att.masked_fill(self.causal_mask[:, :, :seq_len, :seq_len] == 0, float("-inf"))

        # 5. APPLY EXTERNAL MASK (Dynamic Padding)
        # If external_mask is (batch, 1, 1, seq_len), it broadcasts across the query dimension
        if external_mask is not None:
            att = att.masked_fill(external_mask == 0, float("-inf"))

        # 6. Softmax & Value aggregation
        att = F.softmax(att, dim=-1)
        y = att @ v  # (batch, n_head, seq_len, head_size)

        # 7. Re-assemble heads
        y = y.transpose(1, 2).contiguous().view(batch_size, seq_len, n_embd)

        return self.c_proj(y)

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp = nn.Sequential(
            nn.Linear(config.n_embd, 4 * config.n_embd),
            nn.GELU(),
            nn.Linear(4 * config.n_embd, config.n_embd),
        )

    def forward(self, x, mask=None):
        # We pass the mask into the attention layer
        # with residual
        x = x + self.attn(self.ln_1(x), external_mask=mask)
        x = x + self.mlp(self.ln_2(x))
        return x


class GPT(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.config = config
        self.token_embedding = nn.Embedding(config.vocab_size, config.n_embd)
        self.position_embedding = nn.Embedding(config.context_len, config.n_embd)
        self.blocks = nn.ModuleList([TransformerBlock(config) for _ in range(config.n_layer)])
        self.ln_f = nn.LayerNorm(config.n_embd)
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size)

    def forward(
        self,
        idx: torch.Tensor,
    ) -> torch.Tensor:
        """_summary_

        :param idx: _description_
        :return: _description_
        """
        batch, seq_len = idx.size()

        # Positions [0, 1, 2, ..., seq_len-1]
        pos = torch.arange(0, seq_len, dtype=torch.long, device=idx.device).unsqueeze(0)

        x = self.token_embedding(idx) + self.position_embedding(pos)

        for block in self.blocks:
            x = block(x)

        x = self.ln_f(x)
        logits = self.lm_head(x)

        return logits

    @torch.no_grad()
    def generate(
        self,
        prompt_idx: torch.Tensor,
        max_new_tokens: int,
        temperature: float = 1.0,
    ) -> torch.Tensor:
        """_summary_

        :param prompt_idx: _description_
        :param max_new_tokens: _description_
        :param temperature: _description_, defaults to 1.0
        :return: _description_
        """
        # prompt_idx: [1, seq_len]
        for _ in range(max_new_tokens):
            # crop current context to configured context_len
            idx_cond = prompt_idx[:, -self.config.context_len :]

            # get predictions for the current sequence
            logits = self(idx_cond)

            # pull only the last token prediction
            last_logit = logits[:, -1, :]

            # temperature scaling:
            # if T is high, logits become smaller/closer together
            # if T is low, gap beween large and small logits grows wider
            last_logit = last_logit / temperature

            # softmax: convert to probabilities
            probs = F.softmax(last_logit, dim=-1)

            # sampling: (creativity) sample the next token (adds some variety compared to simple argmax)
            # multinomial is like a weighted dice
            # altnerative is argmax, which is just pick index with higest probability
            idx_next = torch.multinomial(probs, num_samples=1)

            # append to the sequence
            prompt_idx = torch.cat((prompt_idx, idx_next), dim=1)

        return prompt_idx